# 02 — Building the enterprise search engine

> **Applied Labs** · **Domain**: RAG · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/12_enterprise_search/02_build.ipynb)

**Prerequisites**:
- [01_architecture.ipynb](./01_architecture.ipynb) — Pipeline architecture, chunking, hybrid retrieval design

**What you will learn**:
- Build a realistic 20+ document synthetic enterprise knowledge base
- Implement semantic chunking with overlap
- Create a dual index: OpenAI embeddings (dense) + BM25 (sparse)
- Combine dense and sparse retrieval via reciprocal rank fusion
- Build a full RAG pipeline with cited, grounded answers
- Run an end-to-end demo with diverse query types

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "numpy>=1.24.0" "pandas>=2.0.0"

import os, re, json, math, textwrap, hashlib, time
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set
from collections import Counter, defaultdict

# OpenAI client — set your API key
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY", "sk-REPLACE-ME"))
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — Synthetic knowledge base

We create a **realistic** enterprise knowledge base with 24 documents spanning
four source types: wiki, chat, docs, and email. Topics cover HR policies,
engineering runbooks, product specs, finance, support tickets, and more.

This simulates the heterogeneous knowledge landscape of a mid-size company.

In [ ]:
@dataclass
class Document:
    """An enterprise knowledge-base document."""
    doc_id: str
    title: str
    source: str
    content: str
    tags: List[str] = field(default_factory=list)
    acl_groups: Set[str] = field(default_factory=lambda: {"all_employees"})

# --- 24-document enterprise corpus ---
KNOWLEDGE_BASE: List[Document] = [
    # --- Wiki (policies & guides) ---
    Document("W01", "Return & Exchange Guidelines", "wiki",
        "All merchandise may be returned within 30 days of purchase. Exchanges follow the same "
        "30-day window. Customers must present the original receipt for a full refund. Items "
        "without receipt qualify for store credit only. Defective items qualify for full "
        "replacement regardless of time elapsed. Digital products are non-refundable after download.",
        ["returns", "policy", "customer-service"]),
    Document("W02", "Employee PTO Policy 2024", "wiki",
        "Full-time employees earn 15 days paid time off per year. PTO accrues at 1.25 days per "
        "month. Unused PTO rolls over up to 5 days into the next calendar year. Requests must be "
        "submitted at least 2 weeks in advance via the HR portal. Manager approval is required. "
        "PTO cannot be used during company-wide blackout periods (Dec 20-31).",
        ["hr", "pto", "benefits"]),
    Document("W03", "Security Incident Response Plan", "wiki",
        "Upon detection of a security incident: 1) Contain the threat immediately — isolate "
        "affected systems. 2) Notify the security team lead within 15 minutes. 3) Begin "
        "investigation — preserve logs, identify attack vector. 4) Remediate — patch "
        "vulnerabilities, rotate credentials. 5) Post-mortem within 48 hours. Escalation "
        "contacts are listed in Appendix A. All incidents must be logged in the SIEM dashboard.",
        ["security", "incident-response"]),
    Document("W04", "Remote Work Policy", "wiki",
        "Employees may work remotely up to 3 days per week with manager approval. Core hours "
        "are 10am-3pm local time for meetings. Home office stipend: $500 one-time for new hires. "
        "VPN must be active for all remote access. Sensitive data must not be accessed on public "
        "WiFi without VPN. Quarterly in-person team days are mandatory.",
        ["hr", "remote-work", "policy"]),
    Document("W05", "Onboarding Checklist — Engineering", "wiki",
        "Day 1: Laptop setup, VPN access, Slack workspace invitation, GitHub org access. "
        "Day 2: Codebase walkthrough with tech lead, dev environment setup. Day 3: First PR — "
        "fix a 'good-first-issue' ticket. Week 1: Shadow on-call rotation. Week 2: Pair "
        "programming on a feature ticket. Month 1: Complete security training and code review "
        "certification.",
        ["onboarding", "engineering"]),
    Document("W06", "Data Retention & Deletion Policy", "wiki",
        "Customer data is retained for 3 years after account closure. PII is anonymized after "
        "1 year of inactivity. Deletion requests (GDPR/CCPA) must be processed within 30 days. "
        "Backup data follows the same retention schedule. Audit logs are retained for 7 years "
        "for compliance purposes. Engineering must verify deletion across all replicas.",
        ["compliance", "data", "gdpr"],
        {"all_employees", "legal_team"}),

    # --- Docs (specs, runbooks, processes) ---
    Document("D01", "Kubernetes Deployment Runbook", "docs",
        "Deploy services to the production k8s cluster using Helm charts. Ensure namespace "
        "isolation between staging and production. Pre-deploy checklist: 1) All tests pass in "
        "CI. 2) Docker image tagged and pushed. 3) Config maps updated. 4) Run canary deploy "
        "to 5% traffic. 5) Monitor error rates for 15 minutes. Rollback: helm rollback "
        "<release> to previous revision.",
        ["devops", "kubernetes", "deployment"],
        {"engineering", "devops"}),
    Document("D02", "Product Spec: Widget Pro v2", "docs",
        "Widget Pro v2 features: 20% faster processing via optimized query engine. New REST "
        "API with OpenAPI 3.0 spec. Redesigned dashboard with real-time analytics. Target "
        "release: Q4 2024. Dependencies: auth service v3, billing service v2. Performance "
        "target: p99 latency < 200ms. Breaking changes: deprecated v1 endpoints removed.",
        ["product", "spec", "widget-pro"]),
    Document("D03", "API Rate Limiting Guide", "docs",
        "All public APIs enforce rate limits: Free tier: 100 requests/minute. Pro tier: 1000 "
        "requests/minute. Enterprise: custom limits via contract. Rate limit headers: "
        "X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset. Exceeded limits return "
        "HTTP 429 with Retry-After header. Implement exponential backoff in client SDKs.",
        ["api", "engineering", "rate-limiting"]),
    Document("D04", "Q3 Customer Satisfaction Survey Results", "docs",
        "Overall satisfaction: 78% (up from 72% in Q2). Top praise: product reliability (89%), "
        "customer support response time (82%). Top complaints: pricing transparency (54%), "
        "documentation quality (61%). NPS score: 42 (up 5 points). Action items: revamp pricing "
        "page, hire technical writer, add more API examples.",
        ["customer-success", "survey"]),
    Document("D05", "Refund Processing Checklist", "docs",
        "Step 1: Verify refund eligibility per Return & Exchange Guidelines. Step 2: Log refund "
        "request in CRM with reason code. Step 3: Process refund via Stripe payment gateway — "
        "use refund API endpoint. Step 4: Send confirmation email using template 'refund-confirm'. "
        "Step 5: Update customer account status. Processing SLA: 3-5 business days.",
        ["finance", "refunds", "process"]),
    Document("D06", "Database Migration Playbook", "docs",
        "Pre-migration: 1) Take full backup. 2) Test migration script on staging. 3) Schedule "
        "maintenance window (Saturday 2am-6am). Migration: run Flyway scripts in order. Verify: "
        "row counts, foreign key integrity, index performance. Rollback: restore from backup "
        "within maintenance window. Post-migration: monitor query performance for 48 hours.",
        ["engineering", "database", "migration"],
        {"engineering", "devops"}),

    # --- Chat (Slack threads) ---
    Document("C01", "Slack: #support — refund escalation", "chat",
        "@alice: Customer #4521 wants a refund on their annual subscription. They've been a "
        "customer for 2 years. @bob: Checked the policy — subscription refunds go through "
        "finance. Loop in @jane from billing. @jane: Processed the prorated refund ($340). "
        "Customer notified via email. @alice: Thanks! Closing ticket.",
        ["support", "refund", "slack"]),
    Document("C02", "Slack: #engineering — deploy incident", "chat",
        "@dave: Heads up — Widget Pro v1.9.3 deploy caused 5xx errors on /api/widgets endpoint. "
        "Rolled back to v1.9.2. @sarah: Root cause: missing env var WIDGET_CACHE_TTL in new "
        "config. @dave: Added to pre-deploy checklist. Redeploying with fix in 30 min. "
        "@sarah: v1.9.3-fix deployed, error rate back to baseline.",
        ["engineering", "incident", "slack"],
        {"engineering"}),
    Document("C03", "Slack: #product — feature request discussion", "chat",
        "@pm_lead: Multiple enterprise customers asking for SSO support. Current auth is "
        "email/password only. @cto: Agreed, this is Q4 priority. Let's support SAML 2.0 and "
        "OIDC. @pm_lead: Adding to roadmap. Target: Widget Pro v2.1. @security_lead: We'll need "
        "to audit the auth flow — scheduling for next sprint.",
        ["product", "sso", "feature-request"]),
    Document("C04", "Slack: #hr — benefits question", "chat",
        "@new_hire: Hi! When does health insurance start? @hr_rep: Coverage starts on your first "
        "day! You should have received enrollment forms in your onboarding packet. Three plans: "
        "Basic ($0/mo), Standard ($150/mo), Premium ($300/mo). @new_hire: Great, signing up for "
        "Standard. Thanks!",
        ["hr", "benefits", "slack"]),

    # --- Email ---
    Document("E01", "Email: Vendor contract renewal — AWS", "email",
        "From: cfo@company.com\nTo: finance_team\nSubject: AWS Contract Renewal\n\nTeam, the "
        "AWS contract is up for renewal in March. Current spend: $48K/month ($576K/year). Usage "
        "has grown 22% YoY. I've negotiated a 3-year commitment for 15% discount ($489K/year). "
        "Please review usage reports and flag optimization opportunities before we sign.",
        ["finance", "vendor", "aws"],
        {"finance_team", "exec_team"}),
    Document("E02", "Email: Support Ticket #4521 — Billing error", "email",
        "From: support@company.com\nTo: finance_team\nSubject: Billing Error — Customer #4521\n\n"
        "Customer reports double charge on invoice #8832. Amount: $680 (2x $340). Finance "
        "confirmed duplicate transaction caused by payment gateway timeout + retry. Refund of "
        "$340 issued. Root cause: missing idempotency key in billing service. Engineering ticket "
        "BIL-2847 created.",
        ["support", "billing", "email"]),
    Document("E03", "Email: Q4 Board Meeting Prep", "email",
        "From: ceo@company.com\nTo: exec_team\nSubject: Q4 Board Deck — Action Items\n\n"
        "Please prepare the following for the Nov 15 board meeting: 1) Revenue forecast (CFO). "
        "2) Product roadmap update (CTO). 3) Customer growth metrics (VP Sales). 4) Engineering "
        "headcount plan (VP Eng). Deck draft due Nov 1.",
        ["executive", "board", "planning"],
        {"exec_team"}),
    Document("E04", "Email: Customer success — Enterprise onboarding", "email",
        "From: cs_lead@company.com\nTo: cs_team\nSubject: Acme Corp Onboarding Kickoff\n\n"
        "Acme Corp ($250K ARR) onboarding starts Monday. Key contacts: CTO (technical), VP Ops "
        "(business). Priorities: SSO integration, data migration from legacy system, custom "
        "dashboard setup. Timeline: 6 weeks. Weekly check-ins every Tuesday 2pm.",
        ["customer-success", "onboarding"]),

    # --- More docs for depth ---
    Document("D07", "Performance Review Guidelines", "docs",
        "Reviews occur semi-annually (June, December). Rating scale: 1 (Needs Improvement) to "
        "5 (Exceptional). Self-review due 2 weeks before review date. Manager calibration "
        "sessions ensure consistency. Promotion criteria: consistent 4+ rating for 2 cycles, "
        "demonstrated leadership, scope expansion. Compensation adjustments tied to review cycle.",
        ["hr", "performance", "reviews"],
        {"all_employees"}),
    Document("D08", "Expense Reimbursement Policy", "docs",
        "Eligible expenses: travel, meals (up to $75/day), client entertainment (pre-approved), "
        "conferences, professional development. Submit receipts within 30 days via Expensify. "
        "Manager approval required for expenses over $200. Reimbursement processed within 2 "
        "pay cycles. Corporate card holders: reconcile monthly.",
        ["finance", "expenses", "policy"]),
    Document("W07", "Code Review Standards", "wiki",
        "All PRs require at least one approval before merge. Reviewers should check: 1) "
        "Correctness — does the code do what it claims? 2) Tests — adequate coverage for new "
        "code paths. 3) Security — no hardcoded secrets, SQL injection, XSS. 4) Performance — "
        "no N+1 queries, appropriate caching. 5) Readability — clear naming, comments for "
        "complex logic. Auto-merge enabled for dependabot PRs with passing CI.",
        ["engineering", "code-review"],
        {"engineering"}),
    Document("W08", "Company Holiday Calendar 2024", "wiki",
        "Paid holidays: New Year's Day (Jan 1), MLK Day (Jan 15), Presidents Day (Feb 19), "
        "Memorial Day (May 27), Independence Day (Jul 4), Labor Day (Sep 2), Thanksgiving "
        "(Nov 28-29), Christmas (Dec 25). Company shutdown: Dec 23-Jan 1. Floating holidays: "
        "2 days, usable anytime with 1-week notice.",
        ["hr", "holidays", "calendar"]),
]

# Summary
source_counts = Counter(doc.source for doc in KNOWLEDGE_BASE)
tag_counts = Counter(tag for doc in KNOWLEDGE_BASE for tag in doc.tags)

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents\n")
print("By source:")
for source, count in source_counts.most_common():
    print(f"  {source:<8} {count:>3} docs")
print(f"\nTop 10 tags:")
for tag, count in tag_counts.most_common(10):
    print(f"  {tag:<20} {count}")

## Section 2 — Chunking pipeline

We implement **semantic chunking with overlap**: split on paragraph boundaries
(double newlines or sentence clusters), then add a configurable overlap window
so context isn't lost at chunk boundaries.

In [ ]:
@dataclass
class Chunk:
    """A text chunk with metadata for retrieval."""
    chunk_id: str
    doc_id: str
    doc_title: str
    source: str
    text: str
    index: int


def semantic_chunk(
    doc: Document,
    max_chunk_chars: int = 400,
    overlap_chars: int = 80,
) -> List[Chunk]:
    """Chunk a document using paragraph boundaries with overlap.

    Args:
        doc: Source document.
        max_chunk_chars: Maximum characters per chunk.
        overlap_chars: Character overlap between consecutive chunks.

    Returns:
        List of Chunk objects.
    """
    # Split on double newlines first, then on sentence boundaries for long paras
    paragraphs = re.split(r'\n\n+', doc.content.strip())
    sentences: List[str] = []
    for para in paragraphs:
        sents = re.split(r'(?<=[.!?])\s+', para.strip())
        sentences.extend([s.strip() for s in sents if s.strip()])

    chunks: List[Chunk] = []
    current_text = ""
    idx = 0

    for sent in sentences:
        if len(current_text) + len(sent) + 1 > max_chunk_chars and current_text:
            chunks.append(Chunk(
                chunk_id=f"{doc.doc_id}-c{idx}",
                doc_id=doc.doc_id,
                doc_title=doc.title,
                source=doc.source,
                text=current_text.strip(),
                index=idx,
            ))
            # Overlap: keep tail of current chunk
            if overlap_chars > 0:
                current_text = current_text[-overlap_chars:].lstrip() + " " + sent
            else:
                current_text = sent
            idx += 1
        else:
            current_text = (current_text + " " + sent).strip()

    if current_text.strip():
        chunks.append(Chunk(
            chunk_id=f"{doc.doc_id}-c{idx}",
            doc_id=doc.doc_id,
            doc_title=doc.title,
            source=doc.source,
            text=current_text.strip(),
            index=idx,
        ))
    return chunks


# Chunk the entire knowledge base
ALL_CHUNKS: List[Chunk] = []
for doc in KNOWLEDGE_BASE:
    doc_chunks = semantic_chunk(doc, max_chunk_chars=400, overlap_chars=80)
    ALL_CHUNKS.extend(doc_chunks)

print(f"Total chunks: {len(ALL_CHUNKS)} from {len(KNOWLEDGE_BASE)} documents\n")
print(f"{'Doc ID':<8} {'Chunks':>6}  {'Title'}")
print("-" * 60)
doc_chunk_counts = Counter(c.doc_id for c in ALL_CHUNKS)
for doc in KNOWLEDGE_BASE:
    print(f"{doc.doc_id:<8} {doc_chunk_counts[doc.doc_id]:>6}  {doc.title[:45]}")

lengths = [len(c.text) for c in ALL_CHUNKS]
print(f"\nChunk length stats: mean={np.mean(lengths):.0f}, "
      f"min={min(lengths)}, max={max(lengths)}, median={np.median(lengths):.0f}")

## Section 3 — Embedding & indexing

We build a **dual index**:
1. **Dense index** — OpenAI `text-embedding-3-small` embeddings + cosine similarity
2. **Sparse index** — BM25 for exact keyword matching

Both are stored in memory (numpy arrays) for simplicity. In production you'd
use a vector database (Pinecone, Weaviate, Qdrant) and an inverted index (Elasticsearch).

In [ ]:
def get_embeddings(texts: List[str], model: str = EMBED_MODEL) -> np.ndarray:
    """Get OpenAI embeddings for a batch of texts.

    Args:
        texts: List of strings to embed.
        model: OpenAI embedding model name.

    Returns:
        numpy array of shape (len(texts), embedding_dim).
    """
    response = client.embeddings.create(input=texts, model=model)
    return np.array([e.embedding for e in response.data])


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between query vector(s) and corpus matrix."""
    a_norm = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-10)
    b_norm = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-10)
    return a_norm @ b_norm.T


# Embed all chunks
print("Embedding chunks... ", end="")
chunk_texts = [c.text for c in ALL_CHUNKS]

# Batch in groups of 50 to respect API limits
BATCH_SIZE = 50
all_embeddings: List[np.ndarray] = []
for i in range(0, len(chunk_texts), BATCH_SIZE):
    batch = chunk_texts[i:i + BATCH_SIZE]
    emb = get_embeddings(batch)
    all_embeddings.append(emb)

CHUNK_EMBEDDINGS = np.vstack(all_embeddings)
print(f"done! Shape: {CHUNK_EMBEDDINGS.shape}")


# --- BM25 index ---
def tokenize(text: str) -> List[str]:
    """Lowercase tokenizer."""
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25Index:
    """BM25 sparse retrieval index."""

    def __init__(self, chunks: List[Chunk], k1: float = 1.5, b: float = 0.75):
        self.chunks = chunks
        self.k1, self.b = k1, b
        self.doc_tokens = [tokenize(c.text) for c in chunks]
        self.avgdl = np.mean([len(t) for t in self.doc_tokens])
        self.N = len(chunks)
        self.df: Dict[str, int] = defaultdict(int)
        for tokens in self.doc_tokens:
            for tok in set(tokens):
                self.df[tok] += 1

    def score(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """Return (chunk_index, score) tuples sorted by score."""
        q_tokens = tokenize(query)
        scores: List[float] = []
        for i, doc_toks in enumerate(self.doc_tokens):
            tf_map = Counter(doc_toks)
            dl = len(doc_toks)
            s = 0.0
            for qt in q_tokens:
                tf = tf_map.get(qt, 0)
                df = self.df.get(qt, 0)
                idf = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)
                num = tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                s += idf * num / den
            scores.append(s)
        ranked = sorted(enumerate(scores), key=lambda x: -x[1])
        return ranked[:top_k]


bm25_index = BM25Index(ALL_CHUNKS)
print(f"BM25 index built: {bm25_index.N} chunks, vocab size {len(bm25_index.df)}")

## Section 4 — Hybrid retrieval

We combine dense and sparse retrieval using **Reciprocal Rank Fusion (RRF)**.
This ensures we get the benefits of both:
- Dense captures semantic similarity (synonym handling)
- BM25 captures exact keyword matches (technical terms, IDs)

In [ ]:
def dense_retrieve(query: str, top_k: int = 10) -> List[Tuple[int, float]]:
    """Retrieve top-k chunks using dense embeddings."""
    q_emb = get_embeddings([query])
    sims = cosine_similarity(q_emb, CHUNK_EMBEDDINGS)[0]
    top_indices = np.argsort(-sims)[:top_k]
    return [(int(idx), float(sims[idx])) for idx in top_indices]


def hybrid_retrieve(
    query: str,
    top_k: int = 5,
    dense_k: int = 15,
    sparse_k: int = 15,
    rrf_k: int = 60,
) -> List[Tuple[Chunk, float, str]]:
    """Hybrid retrieval: dense + BM25 with reciprocal rank fusion.

    Args:
        query: User's search query.
        top_k: Number of final results.
        dense_k: Dense retrieval candidates.
        sparse_k: Sparse retrieval candidates.
        rrf_k: RRF smoothing constant.

    Returns:
        List of (Chunk, rrf_score, retrieval_method) tuples.
    """
    # Dense retrieval
    dense_results = dense_retrieve(query, dense_k)
    # Sparse retrieval
    sparse_results = bm25_index.score(query, sparse_k)

    # RRF fusion
    rrf_scores: Dict[int, float] = defaultdict(float)
    method_map: Dict[int, List[str]] = defaultdict(list)

    for rank, (idx, _) in enumerate(dense_results, 1):
        rrf_scores[idx] += 1.0 / (rrf_k + rank)
        method_map[idx].append("dense")
    for rank, (idx, _) in enumerate(sparse_results, 1):
        rrf_scores[idx] += 1.0 / (rrf_k + rank)
        method_map[idx].append("sparse")

    fused = sorted(rrf_scores.items(), key=lambda x: -x[1])[:top_k]
    results = []
    for idx, score in fused:
        methods = "+".join(method_map[idx])
        results.append((ALL_CHUNKS[idx], score, methods))
    return results


# Test on 5 queries
TEST_QUERIES = [
    "how to handle customer refunds",
    "what is the PTO policy for full-time employees",
    "how to deploy to kubernetes production",
    "what were the Q3 customer satisfaction results",
    "what is the company's data retention policy for GDPR",
]

for query in TEST_QUERIES:
    results = hybrid_retrieve(query, top_k=3)
    print(f"\nQ: {query}")
    for i, (chunk, score, method) in enumerate(results, 1):
        snippet = chunk.text[:80].replace("\n", " ") + "..."
        print(f"  {i}. [{chunk.doc_id}] {chunk.doc_title} (score={score:.4f}, {method})")
        print(f"     {snippet}")

## Section 5 — Answer generation

Now we wire retrieval into a **RAG pipeline**: retrieve relevant chunks, format
them as context, and prompt the LLM to generate a grounded answer with inline
citations. The system prompt enforces faithfulness and abstention.

In [ ]:
SYSTEM_PROMPT = """You are an enterprise knowledge assistant. Answer the user's question
using ONLY the provided context chunks. Follow these rules:

1. CITE every factual claim with [n] where n is the chunk number.
2. If the context doesn't contain enough information, say "I don't have enough
   information to fully answer this" and explain what's missing.
3. If sources conflict, note the discrepancy and present both with citations.
4. Never fabricate information not in the context.
5. Be concise: 2-5 sentences for simple questions, more for complex ones.
"""


def rag_answer(
    query: str,
    top_k: int = 5,
    model: str = CHAT_MODEL,
    verbose: bool = True,
) -> Dict:
    """Full RAG pipeline: retrieve → generate with citations.

    Args:
        query: The user's question.
        top_k: Number of chunks to retrieve.
        model: Chat model for generation.
        verbose: Print intermediate steps.

    Returns:
        Dict with keys: query, answer, citations, model, latency_ms.
    """
    t0 = time.time()

    # Step 1: Retrieve
    retrieved = hybrid_retrieve(query, top_k=top_k)
    t_retrieve = time.time() - t0

    # Step 2: Build prompt
    context_parts: List[str] = []
    citations: List[Dict] = []
    for i, (chunk, score, method) in enumerate(retrieved, 1):
        context_parts.append(f"[{i}] (Source: {chunk.doc_title}, ID: {chunk.chunk_id})\n{chunk.text}")
        citations.append({
            "ref": i,
            "chunk_id": chunk.chunk_id,
            "doc_id": chunk.doc_id,
            "title": chunk.doc_title,
            "source": chunk.source,
            "score": round(score, 4),
            "snippet": chunk.text[:100],
        })

    context_block = "\n\n".join(context_parts)
    user_msg = f"Context:\n{context_block}\n\nQuestion: {query}\n\nAnswer with citations [n]:"

    # Step 3: Generate
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.1,
        max_tokens=500,
    )
    answer = response.choices[0].message.content
    t_total = time.time() - t0

    result = {
        "query": query,
        "answer": answer,
        "citations": citations,
        "model": model,
        "latency_ms": {
            "retrieve": round(t_retrieve * 1000),
            "total": round(t_total * 1000),
            "generate": round((t_total - t_retrieve) * 1000),
        },
    }

    if verbose:
        print(f"Q: {query}\n")
        print(f"A: {answer}\n")
        print("Sources:")
        for c in citations:
            print(f"  [{c['ref']}] {c['title']} ({c['source']}, {c['doc_id']})")
        print(f"\n⏱  Retrieve: {result['latency_ms']['retrieve']}ms | "
              f"Generate: {result['latency_ms']['generate']}ms | "
              f"Total: {result['latency_ms']['total']}ms")
        print("=" * 70)

    return result


# Test with first query
print("Testing RAG pipeline...\n")
result = rag_answer("What is the process for handling customer refunds?")

## Section 6 — End-to-end demo

Let's run 5 diverse queries to exercise different capabilities:
1. **Factual lookup** — Single-doc answer
2. **Multi-source synthesis** — Answer requires multiple documents
3. **Comparison** — Comparing two policies or processes
4. **Multi-hop** — Answer requires chaining facts from different sources
5. **Abstention** — Query about something not in our knowledge base

In [ ]:
DEMO_QUERIES = [
    # 1. Factual lookup
    "How many days of PTO do full-time employees get and what's the rollover limit?",
    # 2. Multi-source synthesis
    "What happened with customer #4521's billing issue and how was it resolved?",
    # 3. Comparison
    "What are the differences between the remote work policy and the PTO policy in terms of approval requirements?",
    # 4. Multi-hop reasoning
    "We're onboarding Acme Corp — what SSO features are planned and when will they be available?",
    # 5. Abstention
    "What is the company's policy on cryptocurrency investments for the corporate treasury?",
]

all_results: List[Dict] = []
for i, query in enumerate(DEMO_QUERIES, 1):
    print(f"\n{'━' * 70}")
    print(f"DEMO QUERY {i}")
    print(f"{'━' * 70}")
    result = rag_answer(query)
    all_results.append(result)

# Summary table
print(f"\n\n{'━' * 70}")
print("DEMO SUMMARY")
print(f"{'━' * 70}")
df_summary = pd.DataFrame([
    {
        "Query": r["query"][:50] + "...",
        "Sources": len(r["citations"]),
        "Retrieve ms": r["latency_ms"]["retrieve"],
        "Generate ms": r["latency_ms"]["generate"],
        "Total ms": r["latency_ms"]["total"],
    }
    for r in all_results
])
print(df_summary.to_string(index=False))

## 🏋️ Exercise 1 — Add document-level access control

Modify `hybrid_retrieve` to accept a `user_groups: Set[str]` parameter and
filter out chunks from documents the user isn't authorized to access. Test with
a query that should return both public and restricted results, and verify
restricted docs are filtered for a non-privileged user.

In [ ]:
# YOUR CODE HERE
# def hybrid_retrieve_with_acl(
#     query: str,
#     user_groups: Set[str],
#     top_k: int = 5,
# ) -> List[Tuple[Chunk, float, str]]:
#     ...

## 🏋️ Exercise 2 — Implement query expansion

Before retrieval, use the LLM to **expand the query** with synonyms and related
terms. For example, "PTO policy" → "PTO paid time off vacation leave policy".
Compare retrieval results with and without expansion on 3 test queries.

In [ ]:
# YOUR CODE HERE
# def expand_query(query: str) -> str:
#     """Use LLM to expand a query with synonyms and related terms."""
#     ...

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | A **24-document synthetic corpus** across 4 sources is enough to demonstrate real enterprise search challenges |
| 2 | **Semantic chunking** with overlap preserves context across boundaries while keeping chunks retrievable |
| 3 | **Dual indexing** (dense embeddings + BM25) is straightforward to implement with OpenAI + numpy |
| 4 | **Hybrid retrieval with RRF** consistently outperforms either dense or sparse alone |
| 5 | **RAG generation** with strict citation enforcement produces verifiable, trustworthy answers |
| 6 | **Abstention** (knowing when to say "I don't know") is as important as generating correct answers |

## What's Next

In **03_evaluate.ipynb**, we rigorously evaluate our search engine: retrieval
metrics (MRR, NDCG, Recall@k), answer quality (faithfulness, relevance),
citation accuracy, and latency/cost analysis.

## References

1. Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020.
2. OpenAI (2024). *Embeddings Guide*. https://platform.openai.com/docs/guides/embeddings
3. Robertson, S. & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. Foundations and Trends in IR.